[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kulinskij-tech/Synergetics/blob/main/syn_py/syn_08_synchronization.ipynb)


# 08. Synchronization and collective phase dynamics

_Graduate course: Introduction to Synergetics · English draft_

> **Guiding question:** How can weak coupling produce a macroscopic rhythm from heterogeneous oscillators?

**Notebook contract.** Run from top to bottom. All parameters are dimensionless unless stated otherwise. Explain every diagnostic you use; a visually attractive trajectory is not by itself evidence of self-organization.


## Learning outcomes

By the end of this module, you should be able to:

- explain how a compliant common support couples Huygens clocks and movable-base metronomes
- derive and test the locking condition for a reduced pair of phase oscillators
- derive the Kuramoto mean-field form
- compute and interpret the complex order parameter
- separate finite-size coherence from a collective transition


## Prerequisite bridge: phase reduction

A stable oscillator returns to the same closed orbit after one period $T$. Its phase $\theta\in[0,2\pi)$ labels position along that orbit and advances at natural frequency $\omega=2\pi/T$. Weak perturbations mainly shift phase because deviations transverse to the cycle decay.

Phase reduction discards amplitude transients and retains $\dot\theta_i=\omega_i+$ coupling. It is accurate for weak coupling and stable cycles, but can fail near amplitude instabilities, oscillator death, or strong forcing. The complex average $N^{-1}\sum_j e^{i\theta_j}$ is vector addition on the unit circle: random phases cancel, aligned phases reinforce.


## Huygens' clocks: synchronization through a moving support

In 1665 Christiaan Huygens reported the "odd sympathy" of two pendulum clocks carried by a common support: after transients their pendula maintained a nearly opposite phase. A modern classroom relative is a pair of self-sustained metronomes on a light platform that can recoil horizontally. Such metronomes commonly approach in-phase motion, although anti-phase motion can also be stable for other support, damping, and escapement parameters. **Synchronization is therefore not a mysterious direct attraction between pendula.** Each oscillator moves the support; the support acceleration perturbs both oscillators and transfers energy and momentum between them.

A minimal mechanical description, with pendulum angles $q_i$ and support displacement $x$, has the structure

$$ml^2\ddot q_i+b\dot q_i+mgl\sin q_i
=\tau_{\mathrm{esc},i}(q_i,\dot q_i)-ml\ddot x\cos q_i,$$

$$(M+2m)\ddot x+\Gamma\dot x+kx
=-ml\sum_{i=1}^2\left(\ddot q_i\cos q_i-\dot q_i^2\sin q_i\right).$$

The escapement torque replaces dissipated energy and creates a stable limit cycle; without that active element these would be decaying pendula, not clocks or metronomes. The course's local Mathematica archive (`metronome.nb`, `metronome0.nb`, and `Two Metronomes.nb`) develops this common-suspension model and diagnoses synchronization with relative phase. The accompanying videos provide an experimental check. The published notebook does not depend on those local files, so it remains portable to Colab.

Under weak coupling, amplitude deviations relax and the mechanical system reduces to a phase-difference equation

$$\dot\delta=\Delta\omega-2K\sin(\delta-\delta_0),\qquad \delta=\theta_2-\theta_1.$$

Here $\delta_0=0$ represents an in-phase attracting branch and $\delta_0=\pi$ an anti-phase branch. Phase locking exists when $|\Delta\omega|\leq 2K$, with stable lag $\delta^*=\delta_0+\sin^{-1}(\Delta\omega/2K)$. This reduced model cleanly predicts locking and phase slip, but it cannot determine $\delta_0$ from the apparatus: that requires the support and escapement mechanics above.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

def phase_locked_pair(delta0, delta_omega=0.08, K=0.12,
                      omega_mean=1.0, dt=0.02, duration=90.0):
    # Weak-coupling phase model for a self-sustained oscillator pair.
    t = np.arange(0.0, duration + dt, dt)
    theta = np.empty((2, t.size))
    theta[:, 0] = (0.0, -2.35)
    omega = np.array([omega_mean - delta_omega/2,
                      omega_mean + delta_omega/2])
    for n in range(t.size - 1):
        delta = theta[1, n] - theta[0, n]
        interaction = K*np.sin(delta - delta0)
        theta[:, n + 1] = theta[:, n] + dt*np.array([
            omega[0] + interaction,
            omega[1] - interaction,
        ])
    wrapped_delta = np.angle(np.exp(1j*(theta[1] - theta[0])))
    return t, theta, wrapped_delta

cases = [(0.0, "in-phase branch"), (np.pi, "anti-phase branch")]
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharex="col")
for col, (delta0, label) in enumerate(cases):
    t, theta, delta = phase_locked_pair(delta0)
    tail = t > 70
    axes[0, col].plot(t[tail], np.cos(theta[0, tail]), label=r"$\cos\theta_1$")
    axes[0, col].plot(t[tail], np.cos(theta[1, tail]), label=r"$\cos\theta_2$", alpha=0.8)
    axes[0, col].set_title(label)
    axes[0, col].set_ylabel("oscillator signal")
    axes[0, col].legend()
    target = np.angle(np.exp(1j*(delta0 + np.arcsin(0.08/(2*0.12)))))
    axes[1, col].plot(t, delta)
    axes[1, col].axhline(target, color="k", ls="--", label=fr"$\delta^*={target:.2f}$")
    axes[1, col].set(xlabel="time", ylabel="wrapped phase difference")
    axes[1, col].legend()
plt.suptitle("Locking selected by the coupling branch")
plt.tight_layout()
plt.show()


## Kuramoto order parameter

The globally coupled phase model is

$$\dot\theta_i=\omega_i+\frac{K}{N}\sum_j\sin(\theta_j-\theta_i).$$

Define $Z=re^{i\psi}=N^{-1}\sum_j e^{i\theta_j}$. Then

$$\dot\theta_i=\omega_i+Kr\sin(\psi-\theta_i).$$

The many-body interaction is encoded by the collective field $Z$, a particularly clean example of circular causality: phases generate the order parameter, which in turn drives each phase. For a symmetric unimodal frequency density $g$, the infinite-population threshold is $K_c=2/[\pi g(0)]$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

def kuramoto(K, omega, theta0, dt=0.03, steps=3500, burn=1500):
    theta = theta0.copy()
    r_history = []
    for n in range(steps):
        Z = np.mean(np.exp(1j*theta))
        theta += dt*(omega + K*np.abs(Z)*np.sin(np.angle(Z)-theta))
        if n >= burn:
            r_history.append(abs(np.mean(np.exp(1j*theta))))
    return np.mean(r_history), np.std(r_history), theta

N = 400
omega = rng.normal(0, 1, N)
theta0 = rng.uniform(0, 2*np.pi, N)
K_values = np.linspace(0, 4, 25)
stats = np.array([kuramoto(K, omega, theta0)[:2] for K in K_values])
Kc_theory = 2/(np.pi*(1/np.sqrt(2*np.pi)))

plt.errorbar(K_values, stats[:, 0], yerr=stats[:, 1], marker="o", ms=4, capsize=2)
plt.axvline(Kc_theory, color="k", ls="--", label=fr"$K_c={Kc_theory:.2f}$")
plt.xlabel("coupling K")
plt.ylabel("time-averaged coherence r")
plt.title("Finite Kuramoto ensemble")
plt.legend()
plt.show()


## Assessed exercise

Part A: scan detuning $\Delta\omega$ and coupling $K$ for the Huygens/metronome phase model. Classify locking by the long-time drift of $\delta$, map the boundary, and compare it with $|\Delta\omega|=2K$. Explain why this calculation alone cannot decide whether a real support selects in-phase or anti-phase motion. Part B: repeat the Kuramoto sweep for $N=100,400,1600$ using independent frequency samples. Estimate the finite-size baseline below threshold, define an operational $K_c(N)$, and extrapolate cautiously toward the theoretical value.

Your submission must include a claim, the mathematical or physical reason for it, numerical evidence, and one limitation.


## Reading and attribution

**Local course reading:** Kuramoto, *Chemical Oscillations, Waves, and Turbulence*; Mikhailov, *Foundations of Synergetics I*, oscillatory media. The instructor archive adds mechanical derivations and experiments in `metronome.nb`, `metronome0.nb`, `Two Metronomes.nb`, `synchro1.mp4`, and `synchro2.mp4`.

**Verified web reading:**
- [Physics World: The secret of the synchronized pendulums](https://physicsworld.com/a/the-secret-of-the-synchronized-pendulums/)
- [Pantaleone: Synchronization of metronomes](https://doi.org/10.1119/1.1501118)
- [Bennett et al.: Huygens's clocks](https://doi.org/10.1098/rspa.2001.0888)
- [Oliveira and Melo: Huygens synchronization of two clocks](https://doi.org/10.1038/srep11548)
- [Kuramoto's Chemical Oscillations, Waves, and Turbulence](https://doi.org/10.1007/978-3-642-69689-3)

The local books are course-provided sources. Web links are supplementary and were verified on 2026-09-04.
